<div dir=rtl style="text-align: right">

# שיעור: מסווג חתולים מול כלבים
## רשת נוירונים Fully Connected עם NumPy בלבד

**מה נלמד:**
- עיבוד מקדים של תמונות אמיתיות  
- HOG — חילוץ פיצ'רים גיאומטריים מתמונות
- מימוש רשת FC עם backpropagation ב-NumPy
- Adam optimizer
- אימון, הערכה, וניתוח שגיאות

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import urllib.request
import tarfile
import pickle
import os
import time

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 13

# Hebrew text in matplotlib requires reversing the string
h = lambda s: s[::-1]

print("\u2705 ספריות נטענו בהצלחה")
print(f"NumPy: {np.__version__}")

<div dir=rtl style="text-align: right">

## הורדת הנתונים

נשתמש ב-**CIFAR-10** — מאגר תמונות קלאסי עם 60,000 תמונות בגודל 32×32 ב-10 קטגוריות.  
נסנן רק **חתולים** (class 3) ו**כלבים** (class 5).

</div>

In [ ]:
def download_and_load_cifar10(data_dir='cifar10_data'):
    url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
    tar_path = os.path.join(data_dir, 'cifar-10.tar.gz')
    extracted_path = os.path.join(data_dir, 'cifar-10-batches-py')
    
    os.makedirs(data_dir, exist_ok=True)
    
    if not os.path.exists(extracted_path):
        print("מוריד CIFAR-10 (~170MB)...")
        start = time.time()
        def progress(count, block_size, total_size):
            pct = min(count * block_size / total_size * 100, 100)
            elapsed = time.time() - start
            print(f"\r   {pct:.1f}% | {elapsed:.0f}s", end='', flush=True)
        urllib.request.urlretrieve(url, tar_path, reporthook=progress)
        print("\n   מחלץ קבצים...")
        with tarfile.open(tar_path) as tar:
            tar.extractall(data_dir)
        os.remove(tar_path)
        print("   \u2705 הורדה הושלמה!")
    else:
        print("\u2705 CIFAR-10 כבר קיים בדיסק")
    
    X_train_list, y_train_list = [], []
    for i in range(1, 6):
        path = os.path.join(extracted_path, f'data_batch_{i}')
        with open(path, 'rb') as f:
            d = pickle.load(f, encoding='bytes')
        X_train_list.append(d[b'data'])
        y_train_list.extend(d[b'labels'])
    
    X_train = np.concatenate(X_train_list)
    y_train = np.array(y_train_list)
    
    with open(os.path.join(extracted_path, 'test_batch'), 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    X_test = d[b'data']
    y_test = np.array(d[b'labels'])
    
    # Reshape: CIFAR-10 stores channels first -> (N, 3, 32, 32) -> (N, 32, 32, 3)
    X_train = X_train.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.uint8)
    X_test  = X_test.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.uint8)
    
    with open(os.path.join(extracted_path, 'batches.meta'), 'rb') as f:
        meta = pickle.load(f, encoding='bytes')
    class_names = [name.decode() for name in meta[b'label_names']]
    
    return X_train, y_train, X_test, y_test, class_names

X_all_train, y_all_train, X_all_test, y_all_test, class_names = download_and_load_cifar10()

print(f"\n\U0001f4ca CIFAR-10:")
print(f"   Train: {X_all_train.shape}")
print(f"   Test:  {X_all_test.shape}")
print(f"   קטגוריות ({len(class_names)}): {class_names}")

In [ ]:
CAT_IDX = 3   # class index in CIFAR-10
DOG_IDX = 5

def filter_cats_dogs(X, y):
    mask = (y == CAT_IDX) | (y == DOG_IDX)
    X_f = X[mask]
    y_f = (y[mask] == DOG_IDX).astype(np.float32)  # 0 = חתול, 1 = כלב
    return X_f, y_f

X_train_imgs, y_train = filter_cats_dogs(X_all_train, y_all_train)
X_test_imgs,  y_test  = filter_cats_dogs(X_all_test,  y_all_test)

print("\U0001f431\U0001f436 Dataset לאחר סינון:")
print(f"   Train: {len(X_train_imgs):,} תמונות | חתולים: {(y_train==0).sum():,} | כלבים: {(y_train==1).sum():,}")
print(f"   Test:  {len(X_test_imgs):,}  תמונות | חתולים: {(y_test==0).sum():,}  | כלבים: {(y_test==1).sum():,}")
print(f"   גודל תמונה: {X_train_imgs.shape[1]}\u00d7{X_train_imgs.shape[2]} פיקסלים, {X_train_imgs.shape[3]} ערוצי צבע (RGB)")

<div dir=rtl style="text-align: right">

## חקירת הנתונים

לפני שמאמנים — בואו נסתכל על הנתונים!  
נראה דוגמאות מייצגות ונבחן את התפלגות הפיקסלים.

</div>

In [ ]:
# --- דוגמאות אקראיות ---
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle(h('דוגמאות מה-Dataset \u2014 שורה עליונה: חתולים | שורה תחתונה: כלבים'), fontsize=14)

cats_idx = np.where(y_train == 0)[0]
dogs_idx = np.where(y_train == 1)[0]

for col in range(8):
    # cats row
    axes[0, col].imshow(X_train_imgs[cats_idx[col]])
    axes[0, col].axis('off')
    if col == 0:
        axes[0, col].set_ylabel(h('חתולים'), fontsize=12)
    # dogs row
    axes[1, col].imshow(X_train_imgs[dogs_idx[col]])
    axes[1, col].axis('off')
    if col == 0:
        axes[1, col].set_ylabel(h('כלבים'), fontsize=12)

plt.tight_layout()
plt.show()

# --- התפלגות ערכי פיקסלים ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors_rgb = ['red', 'green', 'blue']
channel_names = ['Red', 'Green', 'Blue']

for ax, label, mask in zip(axes, [h('חתולים'), h('כלבים')], [y_train == 0, y_train == 1]):
    sample = X_train_imgs[mask][:500]
    for c, (color, name) in enumerate(zip(colors_rgb, channel_names)):
        ax.hist(sample[:, :, :, c].flatten(), bins=50, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(label, fontsize=13)
    ax.set_xlabel(h('ערך פיקסל (0\u2013255)'))
    ax.legend()

plt.suptitle(h('התפלגות ערכי פיקסלים לפי ערוץ צבע'), fontsize=14)
plt.tight_layout()
plt.show()

print(f"ממוצע train: {X_train_imgs.mean():.1f} | סטיית תקן: {X_train_imgs.std():.1f}")
print(f"ממוצע test:  {X_test_imgs.mean():.1f}  | סטיית תקן: {X_test_imgs.std():.1f}")

<div dir=rtl style="text-align: right">

## HOG — Histogram of Oriented Gradients

לפני שנאמן רשת נוירונים, נחלץ **פיצ'רים** מהתמונות.

פיקסלים גולמיים (3072 מספרים לתמונה 32×32 RGB) הם ייצוג **ישיר מאוד** — כל פיקסל קשור למיקום ספציפי. 
רשת FC קשה לה ללמוד מהם כי היא לא מבינה מרחב.

**HOG** (2005) פותר את זה: במקום פיקסלים, הוא מתאר את **הצורות והקצוות** בתמונה.

### האלגוריתם בשלבים:
1. **גרדיאנטים** — מחשבים הפרשים בין פיקסלים שכנים (כמו Sobel). מקבלים עוצמה וכיוון לכל פיקסל.
2. **תאים (cells)** — מחלקים את התמונה לתאים קטנים (4×4 פיקסלים).  
   לכל תא בונים **היסטוגרמה** של כיווני הגרדיאנטים, משוקללת לפי עוצמה.
3. **בלוקים** — מקבצים 2×2 תאים לבלוק ומנרמלים → יציבות לשינויי תאורה.

### למה זה עובד?
HOG מייצג קצוות וצורות בצורה **invariant** (יחסית) לשינויי עוצמת תאורה.  
חתולים יש להם אוזניים מחודדות, כלבים מאפשרים גיוון גדול יותר — HOG לוכד את ההבדלים.

תמונה 32×32 עם cells של 4×4 → **1764 פיצ'רים** (במקום 3072 פיקסלים גולמיים).

</div>

In [ ]:
def compute_hog(image, pixels_per_cell=4, cells_per_block=2, orientations=9):
    """
    מחשב HOG features לתמונה בודדת.
    
    image: (H, W, 3) uint8
    מחזיר: וקטור פיצ'רים חד-ממדי
    """
    # המרה ל-grayscale ונרמול ל-[0,1]
    gray = (0.299 * image[:,:,0] +
            0.587 * image[:,:,1] +
            0.114 * image[:,:,2]) / 255.0
    
    h_img, w = gray.shape
    
    # -- שלב 1: גרדיאנטים --
    Gx = np.zeros_like(gray)
    Gy = np.zeros_like(gray)
    # הפרשים מרכזיים (central differences)
    Gx[:, 1:-1] = gray[:, 2:] - gray[:, :-2]
    Gx[:, 0]    = gray[:, 1]  - gray[:, 0]
    Gx[:, -1]   = gray[:, -1] - gray[:, -2]
    Gy[1:-1, :] = gray[2:, :] - gray[:-2, :]
    Gy[0, :]    = gray[1, :]  - gray[0, :]
    Gy[-1, :]   = gray[-1, :] - gray[-2, :]
    
    magnitude = np.sqrt(Gx**2 + Gy**2)
    direction = np.arctan2(np.abs(Gy), np.abs(Gx)) * 180 / np.pi  # 0-90 degrees (unsigned)
    
    # -- שלב 2: היסטוגרמות לכל תא --
    n_cells_y = h_img // pixels_per_cell
    n_cells_x = w // pixels_per_cell
    
    cell_hists = np.zeros((n_cells_y, n_cells_x, orientations))
    
    for cy in range(n_cells_y):
        for cx in range(n_cells_x):
            y0, y1 = cy * pixels_per_cell, (cy + 1) * pixels_per_cell
            x0, x1 = cx * pixels_per_cell, (cx + 1) * pixels_per_cell
            cell_mag = magnitude[y0:y1, x0:x1]
            cell_dir = direction[y0:y1, x0:x1]
            hist, _ = np.histogram(cell_dir, bins=orientations,
                                   range=(0, 90), weights=cell_mag)
            cell_hists[cy, cx] = hist
    
    # -- שלב 3: נרמול בלוקים --
    n_blocks_y = n_cells_y - cells_per_block + 1
    n_blocks_x = n_cells_x - cells_per_block + 1
    
    features = []
    for by in range(n_blocks_y):
        for bx in range(n_blocks_x):
            block = cell_hists[by:by+cells_per_block, bx:bx+cells_per_block].flatten()
            norm  = np.sqrt(np.dot(block, block) + 1e-6)
            features.append(block / norm)
    
    return np.concatenate(features)

# בדיקה
sample_img = X_train_imgs[0]
sample_hog = compute_hog(sample_img)
print(f"גודל HOG vector: {sample_hog.shape[0]} פיצ'רים")
print(f"(תמונה 32\u00d732, תאים 4\u00d74 \u2192 8\u00d78 תאים \u2192 7\u00d77 בלוקים \u00d7 4 תאים \u00d7 9 bins = {7*7*4*9})")

In [ ]:
def visualize_hog(image, pixels_per_cell=4, orientations=9):
    """מציג תמונה עם הגרדיאנטים ותצוגת HOG"""
    gray = (0.299*image[:,:,0] + 0.587*image[:,:,1] + 0.114*image[:,:,2]) / 255.0
    
    Gx = np.zeros_like(gray)
    Gy = np.zeros_like(gray)
    Gx[:, 1:-1] = gray[:, 2:] - gray[:, :-2]
    Gy[1:-1, :] = gray[2:, :] - gray[:-2, :]
    magnitude = np.sqrt(Gx**2 + Gy**2)
    direction = np.arctan2(np.abs(Gy), np.abs(Gx)) * 180 / np.pi
    
    n_cells_y = gray.shape[0] // pixels_per_cell
    n_cells_x = gray.shape[1] // pixels_per_cell
    cell_hists = np.zeros((n_cells_y, n_cells_x, orientations))
    for cy in range(n_cells_y):
        for cx in range(n_cells_x):
            y0,y1 = cy*pixels_per_cell,(cy+1)*pixels_per_cell
            x0,x1 = cx*pixels_per_cell,(cx+1)*pixels_per_cell
            hist, _ = np.histogram(direction[y0:y1,x0:x1], bins=orientations,
                                   range=(0,90), weights=magnitude[y0:y1,x0:x1])
            cell_hists[cy,cx] = hist
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    axes[0].imshow(image)
    axes[0].set_title(h('תמונה מקורית'))
    axes[0].axis('off')
    
    axes[1].imshow(gray, cmap='gray')
    axes[1].set_title('Grayscale')
    axes[1].axis('off')
    
    axes[2].imshow(magnitude, cmap='hot')
    axes[2].set_title(h('עוצמת גרדיאנט'))
    axes[2].axis('off')
    
    # HOG cell visualization -- draw dominant gradient direction per cell
    axes[3].imshow(image, alpha=0.4)
    scale = pixels_per_cell / 2
    for cy in range(n_cells_y):
        for cx in range(n_cells_x):
            cy_center = cy * pixels_per_cell + pixels_per_cell // 2
            cx_center = cx * pixels_per_cell + pixels_per_cell // 2
            hist = cell_hists[cy, cx]
            dominant_bin = np.argmax(hist)
            angle_rad = (dominant_bin / orientations) * np.pi
            dx = np.cos(angle_rad) * scale
            dy = np.sin(angle_rad) * scale
            strength = hist[dominant_bin] / (hist.sum() + 1e-6)
            axes[3].plot([cx_center - dx, cx_center + dx],
                         [cy_center - dy, cy_center + dy],
                         color='yellow', linewidth=1 + strength * 2, alpha=0.9)
    axes[3].set_title('HOG')
    axes[3].axis('off')
    
    plt.suptitle(h('שלבי HOG על תמונה אחת'), fontsize=14)
    plt.tight_layout()
    plt.show()

# Show for a cat and a dog
print("חתול:")
visualize_hog(X_train_imgs[cats_idx[2]])
print("כלב:")
visualize_hog(X_train_imgs[dogs_idx[3]])

In [ ]:
def extract_features_batch(images, pixels_per_cell=4):
    """מחלץ HOG מכל התמונות"""
    features = []
    n = len(images)
    for i, img in enumerate(images):
        if i % 1000 == 0:
            print(f"\r   {i}/{n} ({i/n*100:.0f}%)", end='', flush=True)
        features.append(compute_hog(img, pixels_per_cell=pixels_per_cell))
    print(f"\r   {n}/{n} (100%) \u2705")
    return np.array(features)

print("מחלץ פיצ'רים מ-train set...")
X_train_hog = extract_features_batch(X_train_imgs)

print("מחלץ פיצ'רים מ-test set...")
X_test_hog  = extract_features_batch(X_test_imgs)

print(f"\nגודל פיצ'רים \u2014 train: {X_train_hog.shape} | test: {X_test_hog.shape}")

# Normalization -- zero mean, unit std (computed on train only!)
feat_mean = X_train_hog.mean(axis=0)
feat_std  = X_train_hog.std(axis=0) + 1e-8

X_train_norm = (X_train_hog - feat_mean) / feat_std
X_test_norm  = (X_test_hog  - feat_mean) / feat_std

print(f"\nאחרי נרמול \u2014 ממוצע train: {X_train_norm.mean():.4f} | std: {X_train_norm.std():.4f}")

<div dir=rtl style="text-align: right">

## הרשת — ארכיטקטורה

נבנה רשת **Fully Connected** עם:

| שכבה | כניסה | יציאה | הפעלה |
|------|-------|-------|-------|
| FC 1 | 1764 | 512 | ReLU + Dropout |
| FC 2 | 512 | 128 | ReLU + Dropout |
| FC 3 | 128 | 1 | Sigmoid |

**Sigmoid** בסוף: מחזיר הסתברות בין 0 ל-1.  
0 = חתול, 1 = כלב.

**Loss**: Binary Cross-Entropy

$$\mathcal{L} = -\frac{1}{m}\sum_{i=1}^{m}\left[y_i\log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\right]$$

**Optimizer**: Adam — מסתגל אוטומטית לקצב הלמידה של כל משקל.

</div>

In [ ]:
class FCNet:
    """
    רשת Fully Connected -- מימוש NumPy בלבד.
    layer_sizes: רשימת גדלי שכבות, למשל [1764, 512, 128, 1]
    """
    
    def __init__(self, layer_sizes, dropout_rate=0.3, l2_reg=1e-4):
        self.layer_sizes  = layer_sizes
        self.dropout_rate = dropout_rate
        self.l2_reg       = l2_reg
        self.L            = len(layer_sizes) - 1   # מספר שכבות
        
        # -- משקולות: He initialization --
        self.W = []
        self.b = []
        for i in range(self.L):
            n_in, n_out = layer_sizes[i], layer_sizes[i+1]
            self.W.append(np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in))
            self.b.append(np.zeros(n_out))
        
        # -- Adam state --
        self.t  = 0
        self.mW = [np.zeros_like(w) for w in self.W]
        self.mb = [np.zeros_like(b) for b in self.b]
        self.vW = [np.zeros_like(w) for w in self.W]
        self.vb = [np.zeros_like(b) for b in self.b]
    
    # --------------- Forward Pass ---------------
    def forward(self, X, training=True):
        """מעבר קדמי. שומר cache לצורך backprop."""
        self._A = [X]   # A[0] = X, A[1..L] = activations
        self._Z = []    # pre-activations
        self._D = []    # dropout masks
        
        A = X
        for i in range(self.L - 1):           # שכבות hidden
            Z = A @ self.W[i] + self.b[i]     # (m, sizes[i+1])
            A = np.maximum(0, Z)              # ReLU
            
            if training:
                D = (np.random.rand(*A.shape) > self.dropout_rate) / (1 - self.dropout_rate)
                A = A * D
            else:
                D = None
            
            self._Z.append(Z)
            self._A.append(A)
            self._D.append(D)
        
        # שכבת פלט -- sigmoid
        Z_out = A @ self.W[-1] + self.b[-1]           # (m, 1)
        A_out = 1.0 / (1.0 + np.exp(-np.clip(Z_out, -500, 500)))
        self._Z.append(Z_out)
        self._A.append(A_out)
        self._D.append(None)
        
        return A_out.flatten()   # (m,)
    
    # --------------- Loss ---------------
    def loss(self, y_pred, y_true):
        """Binary Cross-Entropy + L2"""
        eps = 1e-9
        bce = -np.mean(y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps))
        l2  = sum(np.sum(w**2) for w in self.W) * self.l2_reg / 2
        return bce + l2
    
    # --------------- Backward Pass ---------------
    def backward(self, y_true):
        """
        מעבר אחורי -- חישוב גרדיאנטים לכל המשקולות.
        חייב להיקרא אחרי forward(training=True).
        """
        m = len(y_true)
        gW = [None] * self.L
        gb = [None] * self.L
        
        # גרדיאנט של BCE לגבי פלט sigmoid: dL/dA_out = (A_out - y) / m
        dA = (self._A[-1] - y_true.reshape(m, 1)) / m    # (m, 1)
        
        for i in range(self.L - 1, -1, -1):
            A_prev = self._A[i]                          # (m, sizes[i])
            
            gW[i] = A_prev.T @ dA + self.l2_reg * self.W[i]
            gb[i] = dA.sum(axis=0)
            
            if i > 0:
                dA = dA @ self.W[i].T                    # (m, sizes[i])
                # ביטול dropout
                if self._D[i-1] is not None:
                    dA = dA * self._D[i-1]
                # נגזרת ReLU
                dA = dA * (self._Z[i-1] > 0)
        
        return gW, gb
    
    # --------------- Adam Update ---------------
    def adam_step(self, gW, gb, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        """עדכון משקולות עם Adam optimizer"""
        self.t += 1
        for i in range(self.L):
            # First moment (mean)
            self.mW[i] = beta1 * self.mW[i] + (1 - beta1) * gW[i]
            self.mb[i] = beta1 * self.mb[i] + (1 - beta1) * gb[i]
            # Second moment (uncentered variance)
            self.vW[i] = beta2 * self.vW[i] + (1 - beta2) * gW[i]**2
            self.vb[i] = beta2 * self.vb[i] + (1 - beta2) * gb[i]**2
            # Bias correction
            mW_hat = self.mW[i] / (1 - beta1**self.t)
            mb_hat = self.mb[i] / (1 - beta1**self.t)
            vW_hat = self.vW[i] / (1 - beta2**self.t)
            vb_hat = self.vb[i] / (1 - beta2**self.t)
            # Update
            self.W[i] -= lr * mW_hat / (np.sqrt(vW_hat) + eps)
            self.b[i] -= lr * mb_hat / (np.sqrt(vb_hat) + eps)
    
    # --------------- Predict ---------------
    def predict_proba(self, X):
        """מחזיר הסתברויות P(כלב)"""
        return self.forward(X, training=False)
    
    def predict(self, X, threshold=0.5):
        """מחזיר תוויות בינאריות (0=חתול, 1=כלב)"""
        return (self.predict_proba(X) >= threshold).astype(int)

print("\u2705 FCNet מוכן")
print(f"ארכיטקטורה: [1764 \u2192 512 \u2192 128 \u2192 1]")
total_params = 1764*512 + 512 + 512*128 + 128 + 128*1 + 1
print(f"סך פרמטרים: {total_params:,}")

In [ ]:
def train(model, X_train, y_train, X_val, y_val,
          epochs=80, batch_size=64, lr=0.001):
    
    n = len(X_train)
    history = {'train_loss': [], 'val_loss': [],
                'train_acc': [],  'val_acc':  []}
    
    best_val_acc = 0
    best_W = None
    best_b = None
    
    for epoch in range(1, epochs + 1):
        # --- ערבוב ---
        idx = np.random.permutation(n)
        Xs, ys = X_train[idx], y_train[idx]
        
        epoch_loss = 0.0
        n_batches  = 0
        
        # --- Mini-batches ---
        for start in range(0, n, batch_size):
            Xb = Xs[start:start+batch_size]
            yb = ys[start:start+batch_size]
            
            y_pred = model.forward(Xb, training=True)
            batch_loss = model.loss(y_pred, yb)
            gW, gb = model.backward(yb)
            model.adam_step(gW, gb, lr=lr)
            
            epoch_loss += batch_loss
            n_batches  += 1
        
        # --- מדדים ---
        train_loss = epoch_loss / n_batches
        
        # Accuracy computed on a subset for speed
        sample = np.random.choice(n, min(2000, n), replace=False)
        train_acc = np.mean(model.predict(X_train[sample]) == y_train[sample].astype(int))
        
        val_proba = model.predict_proba(X_val)
        val_loss  = model.loss(val_proba, y_val)
        val_acc   = np.mean(model.predict(X_val) == y_val.astype(int))
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        # שמירת המודל הטוב ביותר
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_W = [w.copy() for w in model.W]
            best_b = [b.copy() for b in model.b]
        
        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | "
                  f"Loss: {train_loss:.4f} | "
                  f"Train Acc: {train_acc:.1%} | "
                  f"Val Acc: {val_acc:.1%}")
    
    # שחזור המודל הטוב ביותר
    model.W = best_W
    model.b = best_b
    print(f"\n\u2705 אימון הסתיים | הדיוק הטוב ביותר על ה-validation: {best_val_acc:.1%}")
    
    return history

# --- הגדרה ואימון ---
net = FCNet(layer_sizes=[1764, 512, 128, 1], dropout_rate=0.3, l2_reg=1e-4)

print("מתחיל אימון...")
print("-" * 65)
history = train(net, X_train_norm, y_train, X_test_norm, y_test,
                epochs=80, batch_size=64, lr=0.001)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], label=h('Train Loss'), color='steelblue', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'],   label=h('Val Loss'),   color='crimson',   linewidth=2, linestyle='--')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title(h('Loss במהלך האימון'))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], label=h('Train Acc'), color='steelblue', linewidth=2)
axes[1].plot(epochs_range, [a*100 for a in history['val_acc']],   label=h('Val Acc'),   color='crimson',   linewidth=2, linestyle='--')
axes[1].axhline(y=80, color='green', linestyle=':', linewidth=1.5, label='80%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel(h('דיוק (%)'))
axes[1].set_title(h('דיוק במהלך האימון'))
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(40, 100)

plt.tight_layout()
plt.show()

print(f"Train Acc סופי: {history['train_acc'][-1]:.1%}")
print(f"Val Acc סופי:   {history['val_acc'][-1]:.1%}")

In [ ]:
# --- Confusion Matrix ---
y_pred = net.predict(X_test_norm)
y_true = y_test.astype(int)

TP = np.sum((y_pred == 1) & (y_true == 1))
TN = np.sum((y_pred == 0) & (y_true == 0))
FP = np.sum((y_pred == 1) & (y_true == 0))
FN = np.sum((y_pred == 0) & (y_true == 1))

cm = np.array([[TN, FP], [FN, TP]])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot confusion matrix
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels([h('חתול'), h('כלב')], fontsize=13)
axes[0].set_yticklabels([h('חתול'), h('כלב')], fontsize=13)
axes[0].set_xlabel(h('תחזית'), fontsize=13)
axes[0].set_ylabel(h('אמת'), fontsize=13)
axes[0].set_title('Confusion Matrix', fontsize=14)
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, f'{cm[i,j]}', ha='center', va='center',
                     fontsize=18, color='white' if cm[i,j] > cm.max()*0.5 else 'black')
plt.colorbar(im, ax=axes[0])

# Metrics bar chart
accuracy  = (TP + TN) / len(y_true)
precision = TP / (TP + FP + 1e-9)
recall    = TP / (TP + FN + 1e-9)
f1        = 2 * precision * recall / (precision + recall + 1e-9)

metric_names  = ['Accuracy', 'Precision', 'Recall', 'F1']
metric_values = [accuracy, precision, recall, f1]
colors_bar    = ['#4a90d9', '#e67e22', '#2ecc71', '#9b59b6']

bars = axes[1].bar(metric_names, [v*100 for v in metric_values], color=colors_bar, width=0.5)
axes[1].set_ylim(0, 105)
axes[1].set_ylabel(h('ציון (%)'), fontsize=13)
axes[1].set_title(h('מדדי ביצועים'), fontsize=14)
axes[1].axhline(80, color='red', linestyle='--', linewidth=1.5, label='80%')
axes[1].legend()
for bar, val in zip(bars, metric_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1%}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n\U0001f4ca תוצאות על Test Set ({len(y_true):,} תמונות):")
print(f"   Accuracy:  {accuracy:.1%}")
print(f"   Precision: {precision:.1%}")
print(f"   Recall:    {recall:.1%}")
print(f"   F1 Score:  {f1:.1%}")

In [ ]:
y_pred_all = net.predict(X_test_norm)
proba_all  = net.predict_proba(X_test_norm)

correct_mask = (y_pred_all == y_test.astype(int))
error_mask   = ~correct_mask

correct_idx = np.where(correct_mask)[0]
error_idx   = np.where(error_mask)[0]

label_names_heb = {0: h('חתול'), 1: h('כלב')}

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle(h('דוגמאות ניבוי \u2014 שורה עליונה: נכון | שורה תחתונה: שגוי'), fontsize=14)

np.random.shuffle(correct_idx)
np.random.shuffle(error_idx)

for col in range(8):
    # Correct predictions
    idx = correct_idx[col]
    axes[0, col].imshow(X_test_imgs[idx])
    axes[0, col].axis('off')
    p = proba_all[idx]
    pred = int(p >= 0.5)
    axes[0, col].set_title(f"{label_names_heb[pred]}\n{p:.0%}", fontsize=8, color='green')
    
    # Wrong predictions
    idx = error_idx[col]
    axes[1, col].imshow(X_test_imgs[idx])
    axes[1, col].axis('off')
    p = proba_all[idx]
    pred  = int(p >= 0.5)
    truth = int(y_test[idx])
    axes[1, col].set_title(f"{label_names_heb[pred]}\n({label_names_heb[truth]})", fontsize=8, color='red')

# Add row labels
axes[0, 0].set_ylabel(h('\u2705 נכון'), fontsize=11)
axes[1, 0].set_ylabel(h('\u274c שגוי'), fontsize=11)

plt.tight_layout()
plt.show()

print(f"תחת 'שגוי': תחזית המודל (בסוגריים: האמת)")

In [ ]:
proba_cache = net.predict_proba(X_test_norm)   # pre-compute for speed

out = widgets.Output()

def show_single(idx):
    img   = X_test_imgs[idx]
    truth = int(y_test[idx])
    prob  = proba_cache[idx]
    pred  = int(prob >= 0.5)
    
    with out:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        
        # Image
        axes[0].imshow(img)
        axes[0].axis('off')
        title_color = 'green' if pred == truth else 'red'
        status = '\u2705' if pred == truth else '\u274c'
        axes[0].set_title(f"{status}  #{idx}", fontsize=14, color=title_color)
        
        # Probability bar
        cat_prob = 1 - prob
        dog_prob = prob
        colors_bar = ['#ff9a9e', '#a8edea']
        axes[1].bar([h('חתול'), h('כלב')], [cat_prob, dog_prob], color=colors_bar, width=0.4, edgecolor='gray')
        axes[1].set_ylim(0, 1.15)
        axes[1].set_ylabel(h('הסתברות'))
        pred_str = 'כלב' if pred == 1 else 'חתול'
        truth_str = 'כלב' if truth == 1 else 'חתול'
        axes[1].set_title(h(f'תחזית: {pred_str} | אמת: {truth_str}'), fontsize=12)
        axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=1)
        axes[1].text(0, cat_prob + 0.03, f'{cat_prob:.1%}', ha='center', fontsize=13, fontweight='bold')
        axes[1].text(1, dog_prob + 0.03, f'{dog_prob:.1%}', ha='center', fontsize=13, fontweight='bold')
        
        plt.tight_layout()
        plt.show()

slider = widgets.IntSlider(
    value=0, min=0, max=len(X_test_imgs)-1, step=1,
    description='תמונה:', continuous_update=False,
    style={'description_width': 'initial'}, layout=widgets.Layout(width='60%')
)

btn_next  = widgets.Button(description='\u25b6 הבא',  button_style='primary')
btn_prev  = widgets.Button(description='\u25c4 קודם', button_style='')
btn_rand  = widgets.Button(description='\U0001f500 אקראי', button_style='warning')
btn_err   = widgets.Button(description='\u274c שגיאה הבאה', button_style='danger')

error_indices = np.where(~correct_mask)[0]
err_ptr = [0]

def on_slider_change(change):
    show_single(slider.value)
def on_next(b):
    slider.value = min(slider.value + 1, slider.max)
def on_prev(b):
    slider.value = max(slider.value - 1, slider.min)
def on_rand(b):
    slider.value = np.random.randint(0, len(X_test_imgs))
def on_err(b):
    slider.value = int(error_indices[err_ptr[0] % len(error_indices)])
    err_ptr[0] += 1

slider.observe(on_slider_change, names='value')
btn_next.on_click(on_next)
btn_prev.on_click(on_prev)
btn_rand.on_click(on_rand)
btn_err.on_click(on_err)

show_single(0)

display(widgets.VBox([
    widgets.HTML('<h3 style="text-align:right; direction:rtl">\U0001f3ae ניסוי עצמאי \u2014 בדוק את המודל</h3>'),
    slider,
    widgets.HBox([btn_prev, btn_next, btn_rand, btn_err]),
    out
]))